In [58]:
import os

os.environ['KERAS_BACKEND'] = 'tensorflow'

import re
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import tensorflow as tf
import keras
from keras import layers
from keras.applications import efficientnet
from keras.layers import TextVectorization

from typing import List, Tuple, Dict, Iterable, Optional
import torch
import torch.nn.functional as F
from PIL import Image

keras.utils.set_random_seed(2806)

# Load data

In [41]:
file_path = "../input/curated-cxr-report-generation-dataset/NLP_aug_datasets/df_train_aug.csv"
df = pd.read_csv(file_path, sep = ",")
df

,id,text,path,aug_text
0,s53865364,"In comparison with the study of ___, there is ...",../input/curated-cxr-report-generation-dataset...,['there is no evidence of pneumothorax with th...
1,s56124320,AP chest compared to ___: PICC line ends in th...,../input/curated-cxr-report-generation-dataset...,['AP chest compared to ___: PICC line ends in ...
2,s50991033,The endotracheal tube tip now lies approximate...,../input/curated-cxr-report-generation-dataset...,['endotracheal tube tip now lies approximately...
3,s50337281,"The cardiac, mediastinal and hilar contours ar...",../input/curated-cxr-report-generation-dataset...,"['cardiac, mediastinal and hilar contours are ..."
4,s51904641,Comparison to ___. No relevant change. Low lun...,../input/curated-cxr-report-generation-dataset...,['no relevant change. Low lung volumes. Modera...
...,...,...,...,...
49995,s50858163,1. Interval extubation and removal of the naso...,../input/curated-cxr-report-generation-dataset...,['internal extubation and removal of the nasog...
49996,s52131300,The Swan-Ganz has been removed. Pacemaker defi...,../input/curated-cxr-report-generation-dataset...,['the Swan-Ganz has been removed. the defibril...
49997,s53747282,The Swan-Ganz catheter is been removed. The ri...,../input/curated-cxr-report-generation-dataset...,['the right IJ cordis is in place and the hear...
49998,314,Low lung volumes. Normal heart size. The trach...,../input/curated-cxr-report-generation-dataset...,['the trachea is midline. Lungs are clear. No ...


In [42]:
X = df[['id', 'path', 'aug_text']]
X['aug_text'] = X['aug_text'].str.replace('[\'', '').str.replace('\']', '')
X

/var/folders/my/rlq5shr56m35b7lt22r8gnk40000gn/T/ipykernel_81328/2914708441.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['aug_text'] = X['aug_text'].str.replace('[\'', '').str.replace('\']', '')


,id,path,aug_text
0,s53865364,../input/curated-cxr-report-generation-dataset...,there is no evidence of pneumothorax with the ...
1,s56124320,../input/curated-cxr-report-generation-dataset...,AP chest compared to ___: PICC line ends in th...
2,s50991033,../input/curated-cxr-report-generation-dataset...,endotracheal tube tip now lies approximately 4...
3,s50337281,../input/curated-cxr-report-generation-dataset...,"cardiac, mediastinal and hilar contours are no..."
4,s51904641,../input/curated-cxr-report-generation-dataset...,no relevant change. Low lung volumes. Moderate...
...,...,...,...
49995,s50858163,../input/curated-cxr-report-generation-dataset...,internal extubation and removal of the nasogas...
49996,s52131300,../input/curated-cxr-report-generation-dataset...,the Swan-Ganz has been removed. the defibrilla...
49997,s53747282,../input/curated-cxr-report-generation-dataset...,the right IJ cordis is in place and the heart ...
49998,314,../input/curated-cxr-report-generation-dataset...,the trachea is midline. Lungs are clear. No pn...


# Data preprocessing

In [44]:
def load_captions_data(X):

    """
    Loads captions (text) data and maps them to corresponding image paths.

    Arguments:
        X: dataframe

    Returns:
        caption_mapping: Dictionary mapping image names and the corresponding captions
        text_data: List containing all teh available captions
    """

    caption_mapping = {}
    text_data = []
    images_to_skip = set()

    for i in range (0, len(X) - 1):
        
        # Image name and captions are separated using a tab
        img_name, caption = X['path'][i], X['aug_text'][i]

        # We will remove caption that are either too short or too long
        tokens = caption.strip().split()

        if len(tokens) < 5:
            images_to_skip.add(img_name)
            continue

        if img_name not in images_to_skip:
            # We will add a start and an end token to each caption
            caption = "<start> " + caption.strip() + " <end>"
            text_data.append(caption)

        caption_mapping[img_name] = [caption]

    for img_name in images_to_skip:
        if img_name in caption_mapping:
            del caption_mapping[img_name]

    return caption_mapping, text_data

def train_val_split(caption_data, train_size = 0.9):
    """
    Split the captioning dataset into train and validation sets.

    Arguments:
        caption_data (dict): Dictionary containing the mapped caption data
        train_size (float): Fraction of all the full dataset to use as training data
        shuffle (bool): Whether to shuffle the dataset before splitting

    Returns:
        Training and validation datasets as two separated dicts
    """

    # Get the list of all image paths
    all_images = list(caption_data.keys())

    # Shuffle to ensure randomness
    np.random.shuffle(all_images)

    # Split into training and validation sets
    train_size = int(len(caption_data) * train_size)

    training_data = {
        img_name: caption_data[img_name] for img_name in all_images[:train_size]
    }

    validation_data = {
        img_name: caption_data[img_name] for img_name in all_images[train_size:]
    }

    # Return two sets
    return training_data, validation_data


In [49]:
captions_mapping, text_data = load_captions_data(X)

# Split the dataset into training and validation sets
train_data, val_test_data = train_val_split(captions_mapping)

print("Number of training samples: ", len(train_data))

val_data, test_data = train_val_split(val_test_data, train_size = 0.5)

print("Number of validation samples: ", len(val_data))
print("Number of test samples: ", len(test_data))

Number of training samples:  44993
Number of validation samples:  2500
Number of test samples:  2500


# Model

In [50]:
# Desired image dimensions
IMAGE_SIZE = (299, 299)

# Vocabulary size
VOCAB_SIZE = 20000

# Fixed length allowed for any sequence
SEQ_LENGTH = 50

# Dimension for the image embeddings and token embeddings
EMBED_DIM = 512

# Per-layer units in the feed forward network
FF_DIM = 512

# Other training parameters
BATCH_SIZE = 64
EPOCHS = 50
AUTOTUNE = tf.data.AUTOTUNE



In [51]:
# Load the model
if (torch.cuda.is_available()):
    device = torch.device("cuda")
elif (torch.backends.mps.is_available()):
    device = torch.device("mps")
else:
    device = torch.device("cpu")

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load('ViT-B/32', device)

In [59]:
def encode_images(model, preprocess, paths: Iterable[str],
                  device: torch.device, batch_size = 64) -> torch.Tensor:
    
    feats = []
    paths = list(paths)

    for i in range (0, len(paths), batch_size):
        batch_paths = paths[i:i+batch_size]
        imgs = []
        for p in batch_paths:
            img = Image.open(p).convert('RGB')
            imgs.append(preprocess(img))
        images = torch.stack(imgs, dim = 0).to(device)
        imf = model.encode_image(images)
        imf = imf / imf.norm(dim = -1, keepdim = True)
        feats.append(imf)
    return torch.cat(feats, dim = 0)

In [60]:
def encode_texts(model, prompts: Iterable[str],
                 device: torch.device, batch_size = 64) -> torch.Tensor:
    
    prompts = list(prompts)
    feats = []
    for i in range(0,len(prompts), batch_size):
        toks = clip.tokenize(prompts[i:i+batch_size], truncate = True).to(device)
        tf = model.encode_text(toks)
        tf = tf / tf.norm(dim = -1, keepdim = True)
        feats.append(tf)
    return torch.cat(feats, dim = 0)

In [ ]:
from clip_tasks import load_clip, encode_images, rank_images_by_prompt, percent_across_images
import pandas as pd

model, preprocess, device = load_clip("ViT-B/32")
image_embs = encode_images(model, preprocess, X["path"].tolist(), device, batch_size=64)

top = rank_images_by_prompt(model, image_embs, device,
                            prompt="chest x-ray showing pleural effusion", topk=10)
for idx, pct, cos in top:
    print(f"{df['path'].iloc[idx]}  ->  {pct:5.2f}%  (cos={cos:.3f})")


TypeError: rank_images_by_prompt() got an unexpected keyword argument 'logit'